In [9]:
import numpy as np
from matplotlib import pyplot as plt
from matplotlib import animation
import os
import pyvista as pv

In [10]:
## parameters ##
timestamp = '2026-06-20--07_24_40'
solution_types = ['C']
interval = 50
plots = 0
gifs = 1
show_figs = False
show_obs = 1

In [58]:
timestep_vals = np.load(f'./data/plots/{timestamp}/timestep_vals.npy')
xx = np.load(f'./data/plots/{timestamp}/xx.npy')
yy = np.load(f'./data/plots/{timestamp}/yy.npy')
zz = np.load(f'./data/plots/{timestamp}/zz.npy')
obs = np.load(f'./data/plots/{timestamp}/obstacle.npy')
params = np.load(f'./data/plots/{timestamp}/shape_params.npy')

y_slice = np.argmin(np.abs(yy - params[1]))
z_slice = np.argmin(np.abs(zz - params[2]))
Y,Z,X = np.meshgrid(yy,zz,xx, indexing='ij')
fname = f'./data/plots/{timestamp}/C_plots.npy'
plots = np.load(fname)
plotxy = plots[:,z_slice,:,0]
plotxy_zvals = np.zeros(np.shape(plotxy))+zz[z_slice]
plotxz = plots[y_slice,:,:,0]
plotxz_yvals = np.zeros(np.shape(plotxz))+yy[y_slice]

x_gridxy = X[:,z_slice,:]
y_gridxy = Y[:,z_slice,:]

x_gridxz = X[y_slice,:,:]
z_gridxz = Z[y_slice,:,:]

x0 = params[0:3]

In [61]:
print(f'len xx: {len(xx)}, len yy: {len(yy)}, len zz: {len(zz)}')

gridxy = pv.StructuredGrid(x_gridxy, y_gridxy, plotxy_zvals)
gridxy['c'] = plotxy.ravel()

gridxz = pv.StructuredGrid(x_gridxz, plotxz_yvals, z_gridxz)
gridxz['c'] = plotxz.ravel()

cmax = np.max(plots[:,:,:,0])

plotter = pv.Plotter(notebook=False, off_screen=True)
plotter.add_mesh(
    gridxy,
    lighting=False,
    show_edges=True,
    clim=[0,cmax]
)
plotter.add_mesh(
    gridxz,
    lighting=False,
    show_edges=True,
    clim=[0,cmax]
)

plotter.add_mesh(pv.Sphere(radius=params[3],center=x0),color=[100,100,100])

plotter.open_gif("c.gif")

pts = gridxy.points.copy()
for i in np.arange(len(timestep_vals)):
    plotxy = plots[:,z_slice,:,i]
    gridxy["c"] = plotxy.transpose().ravel()

    plotxz = plots[y_slice,:,:,i]
    gridxz["c"] = plotxz.transpose().ravel()

    plotter.write_frame()
plotter.close()

len xx: 80, len yy: 40, len zz: 95


In [ ]:
x = np.arange(-20, 20, 0.5)
y = np.arange(-10, 10, 0.5)
x, y = np.meshgrid(x, y)
r = np.sqrt(x**2 + y**2)
z = 0*r

# Create and structured surface
gridxy = pv.Structuredgrid(x, y, z)
gridxy["c"] = z.ravel()

# Create a plotter object and set the scalars to the Z height
plotter = pv.Plotter(notebook=False, off_screen=True)
plotter.add_mesh(
    gridxy,
    # scalars="Height",
    lighting=False,
    show_edges=True,
    # scalar_bar_args={"title": "Height"},
    clim=[-1, 1],
)

# Open a gif
plotter.open_gif("wave.gif")

pts = gridxy.points.copy()

# Update Z and write a frame for each updated position
nframe = 5
for phase in np.linspace(0, 2 * np.pi, nframe + 1)[:nframe]:
    z = np.sin(r + phase)
    # pts[:, -1] = 0
    # gridxy.points = pts
    gridxy["c"] = z.ravel()

    # Write a frame. This triggers a render.
    plotter.write_frame()

# Closes and finalizes movie
plotter.close()